# 01 — Series Deep Dive

Series is the atomic building block of Pandas. Every DataFrame column is a Series.
This notebook covers internals, vectorized operations, alignment mechanics, and
interview-critical behaviors that separate beginners from professionals.

In [ ]:
import pandas as pd
import numpy as np

print(f"Pandas: {pd.__version__}")
print(f"NumPy:  {np.__version__}")

## 1. Series Creation — dtype inference

In [ ]:
# From list — dtype inferred
salaries = pd.Series([85000, 92000, 78000, 110000, 65000],
                     name='salary',
                     index=['Alice', 'Bob', 'Carol', 'Dave', 'Eve'])
print(salaries)
print(f"dtype: {salaries.dtype}")

In [ ]:
# From dict — keys become index labels
grades = pd.Series({'Alice': 88.5, 'Bob': 91.0, 'Carol': 76.5, 'Dave': 95.0, 'Eve': 82.0},
                   name='grade')
print(grades)

# From scalar — broadcasts
bonus = pd.Series(5000, index=salaries.index, name='bonus')
print("\nBroadcast scalar:")
print(bonus)

In [ ]:
# From numpy array — zero-copy when possible
arr = np.array([1.5, 2.3, 3.1, 4.8, 5.2])
s_from_arr = pd.Series(arr, name='score')
print(s_from_arr)
print(f"shares memory with arr: {np.shares_memory(s_from_arr.values, arr)}")

## 2. Series Attributes

In [ ]:
print(f"name:   {salaries.name}")
print(f"dtype:  {salaries.dtype}")
print(f"size:   {salaries.size}")
print(f"shape:  {salaries.shape}")
print(f"nbytes: {salaries.nbytes}")
print(f"\nindex:  {salaries.index.tolist()}")
print(f"values: {salaries.values}")

In [ ]:
# Index types
s_range  = pd.Series([10, 20, 30])          # RangeIndex (default) — memory-efficient
s_int    = pd.Series([10, 20, 30], index=[5, 10, 15])  # Int64Index
s_str    = pd.Series([10, 20, 30], index=['a', 'b', 'c'])  # Index (object)

print(type(s_range.index))   # RangeIndex
print(type(s_int.index))     # Index (int64)
print(type(s_str.index))     # Index (object)

## 3. Index Alignment — the Core Mental Model

When operating on two Series, Pandas **aligns by index label first**, then computes.
Misaligned labels produce `NaN`. This is one of the most common interview traps.

In [ ]:
# Same index — works as expected
total_comp = salaries + bonus
print("Salary + Bonus:")
print(total_comp)

In [ ]:
# MISMATCHED INDEX — produces NaN
s1 = pd.Series([100, 200, 300], index=['a', 'b', 'c'])
s2 = pd.Series([10, 20, 30],   index=['b', 'c', 'd'])  # 'a' missing, 'd' extra

print("s1 + s2 (mismatched index):")
print(s1 + s2)
# 'a': 100 + NaN = NaN
# 'b': 200 + 10  = 210
# 'c': 300 + 20  = 320
# 'd': NaN + 30  = NaN

In [ ]:
# Fix with fill_value — treats missing as 0
result = s1.add(s2, fill_value=0)
print("s1.add(s2, fill_value=0):")
print(result)

## 4. Boolean Series Operations

In [ ]:
high_earner = salaries > 80000
print(high_earner)
print(f"\nany() — at least one: {high_earner.any()}")
print(f"all() — all true:     {high_earner.all()}")
print(f"sum() — count True:   {high_earner.sum()}")
print(f"mean() — fraction:    {high_earner.mean():.0%}")

## 5. Value Counts

In [ ]:
np.random.seed(42)
departments = pd.Series(
    np.random.choice(['Engineering', 'Sales', 'HR', 'Marketing', None], 100,
                     p=[0.35, 0.25, 0.15, 0.2, 0.05]),
    name='department'
)

# Default: excludes NaN
print(departments.value_counts())

# Include NaN
print("\nWith NaN:")
print(departments.value_counts(dropna=False))

# Normalize to proportions
print("\nNormalized:")
print(departments.value_counts(normalize=True).round(3))

## 6. Unique Values, Duplicates

In [ ]:
ratings = pd.Series([4, 5, 3, 4, 5, 5, 2, 3, 4], name='rating')

print(f"unique():    {ratings.unique()}")       # array, preserves order of first occurrence
print(f"nunique():   {ratings.nunique()}")       # int count
print(f"duplicated():\n{ratings.duplicated()}")  # bool mask — first occurrence = False

In [ ]:
# duplicated with keep='last' — mark all except last
print(ratings.duplicated(keep='last'))

# keep=False — mark ALL duplicates
print("\nAll duplicates marked:")
print(ratings.duplicated(keep=False))

## 7. isin() — Membership Filter

In [ ]:
target_depts = ['Engineering', 'Marketing']
mask = departments.isin(target_depts)
print(departments[mask].value_counts())

## 8. Element Access — loc, iloc, at, iat

In [ ]:
print(f"Label access  salaries['Alice']:    {salaries['Alice']}")
print(f"Positional    salaries.iloc[0]:     {salaries.iloc[0]}")
print(f"Scalar at     salaries.at['Alice']: {salaries.at['Alice']}")
print(f"Scalar iat    salaries.iat[0]:      {salaries.iat[0]}")

# at/iat are faster than loc/iloc for single scalar access
# Use them inside tight loops when you must iterate (but avoid iteration)

In [ ]:
# Slice with loc — label-based, BOTH ends inclusive
print(salaries.loc['Alice':'Carol'])

# Slice with iloc — position-based, end EXCLUSIVE
print("\niloc[0:3]:")  
print(salaries.iloc[0:3])  # Alice, Bob, Carol (not Dave)

## 9. Series.map() and apply()

In [ ]:
# map() with dict — unmapped labels become NaN
dept_code = {'Engineering': 'ENG', 'Sales': 'SAL', 'HR': 'HRM', 'Marketing': 'MKT'}
dept_abbrev = departments.map(dept_code)
print(dept_abbrev.value_counts(dropna=False).head())

In [ ]:
# map() with function
salary_band = salaries.map(lambda x: 'High' if x >= 90000 else 'Standard')
print(salary_band)

In [ ]:
# apply() — use when you need complex logic
# For element-wise on Series, map() is preferred; apply() is same speed here
# apply() shines on DataFrames (axis=1)
def classify_salary(x):
    if x >= 100000: return 'Senior'
    elif x >= 80000: return 'Mid'
    else: return 'Junior'

salary_level = salaries.apply(classify_salary)
print(salary_level)

## 10. String Accessor — str

In [ ]:
cities = pd.Series(['Mumbai', 'Delhi', 'Bangalore', 'Pune', 'Chennai'], name='city')

print(cities.str.upper())
print()
print(cities.str.len())
print()
print(cities.str.contains('a', case=False))  # case-insensitive

In [ ]:
# String methods return Series — chainable
# Extract first 3 characters
print(cities.str[:3])

# Replace
print(cities.str.replace('a', '@', regex=False))

## 11. Cumulative Operations

In [ ]:
monthly_revenue = pd.Series([12000, 15000, 9000, 18000, 21000],
                            index=['Jan', 'Feb', 'Mar', 'Apr', 'May'],
                            name='revenue')

print("Revenue:")
print(monthly_revenue)
print("\nCumulative sum (YTD revenue):")
print(monthly_revenue.cumsum())
print("\nRunning max:")
print(monthly_revenue.cummax())

## 12. describe() — Numeric vs Object

In [ ]:
# Numeric describe
print("Numeric:")
print(salaries.describe())

In [ ]:
# Object/string describe — different stats
print("Object:")
print(cities.describe())

## 13. between() and clip()

In [ ]:
# between() — inclusive on both ends by default
mid_range = salaries.between(75000, 95000)
print("Salaries between 75k-95k:")
print(salaries[mid_range])

# clip() — cap values at lower/upper bounds
capped = salaries.clip(lower=70000, upper=100000)
print("\nClipped salaries (70k–100k):")
print(capped)

## 14. Conversion Methods

In [ ]:
print("to_list():  ", salaries.to_list())
print("to_dict():  ", salaries.to_dict())

# to_frame() — convert Series to single-column DataFrame
salary_df = salaries.to_frame()
print("\nto_frame():")
print(salary_df)

## 15. Memory Usage

In [ ]:
# Default memory_usage — excludes index
# deep=True — follows object references (critical for object dtype)

obj_series = pd.Series(['Engineering', 'Sales', 'HR'] * 1000)
cat_series = obj_series.astype('category')

obj_mem = obj_series.memory_usage(deep=True)
cat_mem = cat_series.memory_usage(deep=True)

print(f"Object dtype:      {obj_mem:,} bytes")
print(f"Category dtype:    {cat_mem:,} bytes")
print(f"Reduction:         {(1 - cat_mem/obj_mem):.1%}")

## 16. Interview: Alignment Trap in Production

Classic mistake: computing ratio between two Series built from different filtered DataFrames.

In [ ]:
np.random.seed(42)
df = pd.DataFrame({
    'employee': [f'E{i:03d}' for i in range(10)],
    'sales_q1': np.random.randint(50000, 200000, 10),
    'sales_q2': np.random.randint(50000, 200000, 10)
})

# Two filtered subsets — different indexes!
top_q1 = df[df.sales_q1 > 100000]['sales_q1']
top_q2 = df[df.sales_q2 > 100000]['sales_q2']

print("Q1 high performers index:", top_q1.index.tolist())
print("Q2 high performers index:", top_q2.index.tolist())

# This aligns by index — NaN for non-matching rows!
ratio = top_q1 / top_q2
print("\nRatio (NaN from misalignment):")
print(ratio)

In [ ]:
# Fix: reset index to force positional alignment (only valid when shapes match)
# OR: operate on the original DataFrame columns directly
df['growth_ratio'] = df['sales_q2'] / df['sales_q1']
print(df[['employee', 'sales_q1', 'sales_q2', 'growth_ratio']].round(3))

## 17. dt Accessor — Preview

In [ ]:
dates = pd.Series(pd.date_range('2023-01-01', periods=6, freq='ME'), name='month_end')
print(dates.dt.year)
print(dates.dt.month_name())
print(dates.dt.day_of_week)  # 0=Monday

## 18. Arithmetic Methods — fill_value Parameter

In [ ]:
# All arithmetic ops have method equivalents with fill_value
s_a = pd.Series({'x': 10, 'y': 20, 'z': 30})
s_b = pd.Series({'y': 5,  'z': 10, 'w': 15})

ops = {
    'add (fill=0)':  s_a.add(s_b, fill_value=0),
    'sub (fill=0)':  s_a.sub(s_b, fill_value=0),
    'mul (fill=1)':  s_a.mul(s_b, fill_value=1),
    'div (fill=1)':  s_a.div(s_b, fill_value=1),
}

for name, result in ops.items():
    print(f"\n{name}")
    print(result)

## 19. Sorting a Series

In [ ]:
# sort_values — ascending/descending
print(salaries.sort_values(ascending=False))

# sort_index
print("\nSorted by index:")
print(salaries.sort_index())

# nlargest / nsmallest — more efficient than sort for top-N
print("\nTop 3 salaries:")
print(salaries.nlargest(3))

## 20. Quick Reference — Important Series Methods

| Method | Use Case |
|--------|----------|
| `s.map(dict/func)` | Element-wise transform, dict mapping |
| `s.apply(func)` | Complex element-wise function |
| `s.value_counts()` | Frequency table |
| `s.isin(list)` | Membership filter |
| `s.between(lo, hi)` | Range filter |
| `s.str.*` | String operations |
| `s.dt.*` | DateTime operations |
| `s.add(s2, fill_value=0)` | Aligned arithmetic with fill |
| `s.cumsum()` | Running total |
| `s.nlargest(n)` | Top-N values |
| `s.clip(lo, hi)` | Cap outliers |
| `s.memory_usage(deep=True)` | True memory footprint |